# DCASE AudioQA Validation Test with Qwen 2.5 and vLLM

This notebook randomly selects validation problems from `/work/u1501463/2025_DCASE_AudioQA_Official/dcase_2025_question_path/dev` and tests them with `Qwen/Qwen2.5-Omni-7B` using `vllm`.

In [1]:
import os
os.environ.pop("TRITON_PTXAS_PATH", None)

from vllm import LLM, SamplingParams
import librosa
import numpy as np
import json
import glob
import random

# llm = LLM(
#     model="Qwen/Qwen2.5-Omni-7B",
#     max_model_len=20000,
#     max_num_seqs=1,
#     limit_mm_per_prompt={"audio": 4},
#     trust_remote_code=True,
# )

INFO 05-11 22:55:03 [__init__.py:239] Automatically detected platform cuda.


In [2]:
def load_audio(path, sr=16000):
    y, _ = librosa.load(path, sr=sr, mono=True)
    return (y.astype(np.float32), sr)

root = "/work/u1501463/2025_DCASE_AudioQA_Official"
validation_json_dir = os.path.join(root, "dcase_2025_question_path", "dev")
audio_root = os.path.join(root, "local_audio_path", "dev")

question_files = sorted(glob.glob(os.path.join(validation_json_dir, "*.json")))
print(f"Found {len(question_files)} validation JSON files")

random.seed(42)
num_samples = 5
selected_files = random.sample(question_files, min(num_samples, len(question_files)))

Found 2466 validation JSON files


In [3]:
from pprint import pprint
N = 30
for p in selected_files:
    with open(p, "r") as f:
        data = json.load(f)
    pprint(data)
    audio_path = os.path.join(audio_root, data["audio_url"])
    # audio, sr = load_audio(audio_path)
    # print(f"Loaded audio from {audio_path} with shape {audio.shape} and sample rate {sr}")

{'answer': 'A. A success or victory',
 'audio_url': '../../local_audio_path/dev/audio_0001409.wav',
 'choice': ['A. A success or victory',
            'B. An announcement',
            'C. A sound effect',
            'D. A technical glitch'],
 'id': 'dev_audio_0001409',
 'question': 'Why did the crowd cheer?',
 'question_type': 'both'}
{'answer': 'B. Groovy bass',
 'audio_url': '../../local_audio_path/dev/audio_01498.wav',
 'choice': ['A. Mellow flute',
            'B. Groovy bass',
            'C. Gentle piano',
            'D. Soft violin'],
 'id': 'dev_audio_01498',
 'question': 'Which instrumental component is prominent during the emotional '
             'vocal delivery?',
 'question_type': 'both'}
{'answer': 'C. True',
 'audio_url': '../../local_audio_path/dev/audio_00287.wav',
 'choice': ['A. False', 'B. cannot be determined', 'C. True'],
 'id': 'dev_audio_00287',
 'question': 'Is it hazardous to have the audio play as a background in '
             'hospital indoors?',
 'quest

In [3]:
import sys
from pathlib import Path
sys.path.append('/home/u1501463/tool_use_LALM/')
import prompts

DEFAULT_SYSTEM = (
    "You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, "
    "capable of perceiving auditory and visual inputs, as well as generating text and speech."
)

def tool_use_QA_prompt(question, choices, tool_use_prompt):
    system_prompt =\
'''{DEFAULT_SYSTEM}
You will given a question and a set of choices. You need to choose the most appropriate answer from the choices based on the question and the tool use prompt. The tool use prompt provides information about how to use the tools to answer the question. You can use the tools to find the answer if needed. If you are not sure about the answer, you can use the tools to gather more information before making a decision.
You will be given:
- A question about an audio clip
- Multiple-choice options
- An audio input
- A list of available tools with descriptions

### Question Information:

Question: {question}
Choices: {choices}
Audio: {audio_token}
Audio id: {audio_id}
Following are the descriptions of the tools you can use:
{tools_description}
'''

In [ ]:
def build_prompt(question: str, choices: list[str]) -> str:
    choice_text = "\n".join(choices) if choices else ""
    return (
        f"<|im_start|>system\n{default_system}<|im_end|>\n"
        "<|im_start|>user\n"
        "<|audio_bos|><|AUDIO|><|audio_eos|>\n"
        f"Question: {question}\n"
        f"Choices:\n{choice_text}\n"
        "Please answer with the best choice letter and the full choice text.\n"
        "<|im_end|>\n"
        "<|im_start|>assistant\n"
    )


def resolve_audio_path(item: dict) -> str:
    audio_url = item.get("audio_url", "")
    audio_path = os.path.normpath(os.path.join(validation_json_dir, audio_url))
    if os.path.exists(audio_path):
        return audio_path
    fallback_path = os.path.normpath(os.path.join(root, audio_url))
    if os.path.exists(fallback_path):
        return fallback_path
    raise FileNotFoundError(f"Audio file not found for URL: {audio_url}")

default_system = (
    "You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, "
    "capable of perceiving auditory and visual inputs, as well as generating text and speech."
)

sampling_params = SamplingParams(
    temperature=0.2,
    max_tokens=256,
)

for i, json_path in enumerate(selected_files, start=1):
    with open(json_path, "r", encoding="utf-8") as f:
        item = json.load(f)

    question = item.get("question", "")
    choices = item.get("choice", [])
    answer = item.get("answer", "N/A")
    audio_path = resolve_audio_path(item)

    audio_data, sr = load_audio(audio_path)
    prompt = build_prompt(question, choices)

    outputs = llm.generate(
        [
            {
                "prompt": prompt,
                "multi_modal_data": {
                    "audio": [(audio_data, sr)],
                },
            }
        ],
        sampling_params=sampling_params,
    )

    pred = outputs[0].outputs[0].text.strip()
    print(f"=== Sample {i} ===")
    print(f"JSON: {json_path}")
    print(f"Question: {question}")
    print("Choices:")
    for choice in choices:
        print(choice)
    print(f"Ground truth: {answer}")
    print("Prediction:")
    print(pred)
    print("\n")